# Kapitel 20.1 - Grundlagen: Concurrency in Python

Dieses Notebook erklaert die Basis von Nebenlaeufigkeit: Prozesse, Threads und asynchrone Coroutines.

# Lernziele

- Unterschied zwischen Parallelitaet und Concurrency erklaeren
- I/O-bound und CPU-bound Aufgaben unterscheiden
- passende Technik fuer ein Problem waehlen

# Voraussetzungen

- Funktionen, Schleifen, Module
- Grundlagen zu Exceptions

# Theorie

Concurrency bedeutet, mehrere Aufgaben ueberlappend zu bearbeiten.
Parallelitaet bedeutet, dass Aufgaben tatsaechlich gleichzeitig auf unterschiedlichen Kernen laufen.

In Python sind wichtige Werkzeuge:
- `threading` fuer I/O-lastige Aufgaben
- `multiprocessing` fuer CPU-lastige Aufgaben
- `asyncio` fuer viele wartende I/O-Operationen

# Erklaerung

Warum ist das wichtig?

Viele Anwendungen warten auf Netzwerk, Datei oder Datenbank. Ohne Concurrency bleibt CPU-Zeit ungenutzt.
Mit der richtigen Strategie werden Programme reaktionsfaehiger und skalierbarer.

# Syntax

```python
import threading
import asyncio

thread = threading.Thread(target=funktion)
thread.start()

async def aufgabe():
    await asyncio.sleep(1)
```

# Merke

- Nicht jede Aufgabe braucht Concurrency.
- Komplexitaet steigt: Debugging und Fehlerbehandlung werden wichtiger.

# Parameter

Beispiele:
- `Thread(target=..., args=...)`
- `asyncio.create_task(coro)`
- `await` fuer nicht-blockierende Wartezeiten

# Rueckgabewert

Threads geben Ergebnis oft indirekt weiter (Queue, gemeinsamer Zustand).
Coroutines liefern direkt Rueckgabewerte via `await`.

In [ ]:
# Beispiel 1: Sequenziell
import time

def arbeit(name):
    print(f'{name} startet')
    time.sleep(1)
    print(f'{name} fertig')

start = time.time()
arbeit('A')
arbeit('B')
print('Dauer:', round(time.time()-start, 2), 'Sekunden')

In [ ]:
# Beispiel 2: Threading
import threading

start = time.time()
t1 = threading.Thread(target=arbeit, args=('A',))
t2 = threading.Thread(target=arbeit, args=('B',))
t1.start(); t2.start()
t1.join(); t2.join()
print('Dauer:', round(time.time()-start, 2), 'Sekunden')

In [ ]:
# Beispiel 3: Asyncio
import asyncio

async def async_arbeit(name):
    print(f'{name} startet')
    await asyncio.sleep(1)
    print(f'{name} fertig')

async def main():
    start = time.time()
    await asyncio.gather(async_arbeit('A'), async_arbeit('B'))
    print('Dauer:', round(time.time()-start, 2), 'Sekunden')

asyncio.run(main())

# Praxisbeispiel

Vergleiche fuer 10 simulierte API-Aufrufe:
- sequenziell
- mit Threads
- mit asyncio

Notiere Laufzeiten und begruende das Ergebnis.

# Haeufige Fehler

1. CPU-lastige Aufgaben mit zu vielen Threads beschleunigen wollen.
2. `await` vergessen (Coroutine wird nicht ausgefuehrt).
3. Shared State ohne Synchronisation nutzen.

# Best Practice

- Erst messen, dann optimieren.
- Nebenlaeufigkeit nur dort einsetzen, wo sie echten Nutzen bringt.
- Kleine, testbare Einheiten bauen.

# Tipp

Starte immer mit einer einfachen sequentiellen Version und erweitere danach zu Threading/Asyncio.

# Uebung

Schreibe ein Programm, das 5 URLs simuliert und die schnellste Implementierung vergleicht.

# Loesung

Nutze `time.sleep` fuer Threading-Simulation und `asyncio.sleep` fuer Coroutine-Simulation.
Messe die Dauer pro Ansatz und dokumentiere die Ergebnisse.

# Zusammenfassung

Du kennst jetzt die Grundprinzipien von Concurrency und kannst geeignete Techniken auswaehlen.

# Weiterfuehrende Links

- Python `threading`
- Python `asyncio`
- Python `multiprocessing`

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

Race Condition
Critical Section
Event Loop
Backpressure
Cancellation
Timeout Budget

In [ ]:
# asyncio Timeout-Guard
import asyncio
async def slow_job():
    await asyncio.sleep(2)
    return "done"
async def main():
    try:
        print(await asyncio.wait_for(slow_job(), timeout=1.0))
    except asyncio.TimeoutError:
        print("timeout")
asyncio.run(main())

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.